# 4차시 실습 노트북
## 교과서 문제
세종과학고등학교 정보과학 · 1학년

## 문제 1 &nbsp;&nbsp;`비선형 탐색`

자연수의 합으로 `n` 을 나타낼 수 있는 모든 순열의 개수를 구해 보자.

`n` 이 3이면 `1+1+1`, `1+2`, `2+1`, `3` 으로 4가지다.

①은 종료 조건이다. `node == 0` 이면 합이 정확히 `n` 이 된 것이므로 1가지로 센다.
순열이므로 `1+2` 와 `2+1` 을 다르게 센다. 그래서 `i` 를 매번 1부터 다시 고른다.

**실행 결과**
```
n = 3
순열의 개수 : 4
```

In [ ]:
def count(node):
    if node == 0:          # ①
        return 1

    total = 0
    for i in range(1, node + 1):        # ②
        total = total + count(node - i)

    return total


n = 3

print('n =', n)
print('순열의 개수 :', count(n))

### 교과서 코드 (교과서 175쪽)

교과서 코드는 `f(0)~f(n-1)` 을 더하는 형태지만 결과는 같다.

```
입력          실행 결과
3             4
```

In [ ]:
def f(n):
    if n==0:
      return 1
    cnt = 0
    for i in range(n):
       cnt = cnt+f(i)
    return cnt

n = int(input())
print(f(n))

### 재귀를 쓰지 않으면 &nbsp;&nbsp;<span style="color:#6b4bc4">반복문 풀이</span>

`f(n)` 은 `f(0)`~`f(n-1)` 을 **전부 더한 값**이다. 앞에서부터 차례로 쌓아 두면 재귀가 필요 없다.
값을 나열해 보면 `1, 1, 2, 4, 8, …` 이므로 `f(n) = 2^(n-1)` 이다.

**실행 결과**
```
n = 3
반복문으로 구한 순열의 개수 : 4
공식 2 ** (n-1)           : 4
```

In [ ]:
n = 3

F = [1]                    # F[0] = 1
for i in range(1, n + 1):
    F.append(sum(F))       # F[i] = F[0] + ... + F[i-1]

print('n =', n)
print('반복문으로 구한 순열의 개수 :', F[n])
print('공식 2 ** (n-1)           :', 2 ** (n - 1))

### 재귀를 살려 두고 고치면 &nbsp;&nbsp;<span style="color:#6b4bc4">동적 계획법(DP) 풀이</span>

교과서 재귀는 **같은 부분 문제를 몇 번이고 다시 푼다**(`f(4)` 를 구하는 동안 `f(0)` 은 8번).
한 번 구한 답을 `memo` 에 **적어 두었다가 다시 쓰면**(메모이제이션) 재귀 구조는 그대로 둔 채 같은 계산을 반복하지 않는다.

| | 하향식 (메모이제이션) | 상향식 (타뷸레이션) |
|---|---|---|
| 모양 | 재귀 + `memo` 딕셔너리 | `for` 문 + 배열 |
| 이 문제에서는 | 아래 `f_memo` | 위의 반복문 풀이 |

**실행 결과**
```
n = 20
메모이제이션으로 구한 순열의 개수 : 524288
```

In [ ]:
memo = {}

def f_memo(n):
    if n == 0:
        return 1
    if n in memo:              # 이미 구한 값이면 다시 계산하지 않는다
        return memo[n]
    cnt = 0
    for i in range(n):
        cnt = cnt + f_memo(i)
    memo[n] = cnt
    return cnt


n = 20

print('n =', n)
print('메모이제이션으로 구한 순열의 개수 :', f_memo(n))

### 재귀를 쓰지 않으면 &nbsp;&nbsp;<span style="color:#6b4bc4">DFS · BFS 풀이</span>

위의 재귀는 **이미 DFS다.** 호출 스택을 직접 만들어 쓰면 3차시의 DFS·BFS가 그대로 적용된다.

```
                    4
        ┌─────┬─────┴──┬─────┐
       3      2        1     0 ✓
     ┌─┼─┐   ┌┴─┐      │
    2  1  0✓ 1  0✓     0✓
   ...      ...
```

**노드** = 남은 합, **간선** = 다음 항 고르기, **`node == 0` 인 잎** = 성공한 순열 하나. 즉 **잎을 세는 문제**다.
두 함수는 `pop()` 과 `popleft()` 하나만 다르다. `node` 가 같아도 경로가 다르면 다른 순열이므로 `visited` 는 쓰지 않는다.

**실행 결과**
```
n = 3
DFS(스택)로 구한 순열의 개수 : 4
BFS(큐)로 구한 순열의 개수   : 4
```

In [ ]:
from collections import deque

def count_dfs(n):
    stack = [n]                     # 아직 살펴보지 않은 상태들
    cnt = 0
    while stack:
        node = stack.pop()          # ① 맨 뒤에서 꺼낸다 (LIFO)
        if node == 0:               # 합이 정확히 n 이 되었다
            cnt = cnt + 1
            continue
        for x in range(1, node + 1):
            stack.append(node - x)
    return cnt


def count_bfs(n):
    q = deque([n])
    cnt = 0
    while q:
        node = q.popleft()          # ② 맨 앞에서 꺼낸다 (FIFO)
        if node == 0:
            cnt = cnt + 1
            continue
        for x in range(1, node + 1):
            q.append(node - x)
    return cnt


n = 3

print('n =', n)
print('DFS(스택)로 구한 순열의 개수 :', count_dfs(n))
print('BFS(큐)로 구한 순열의 개수   :', count_bfs(n))

#### 찾는 순서는 어떻게 다른가

고른 항을 같이 들고 다니면 **찾아내는 순서**가 보인다.
DFS는 사전순, BFS는 항의 개수 순이다(레벨 = 고른 항의 개수). 그래서 BFS가 처음 찾은 답은 항이 가장 적은 답이다.

DFS에서 `range(node, 0, -1)` 로 거꾸로 넣는 것은 스택이 나중에 넣은 것을 먼저 꺼내기 때문이다.

**실행 결과**
```
DFS 순서 : 1+1+1+1 , 1+1+2 , 1+2+1 , 1+3 , 2+1+1 , 2+2 , 3+1 , 4
BFS 순서 : 4 , 1+3 , 2+2 , 3+1 , 1+1+2 , 1+2+1 , 2+1+1 , 1+1+1+1
```

In [ ]:
from collections import deque

def paths_dfs(n):
    out = []
    stack = [(n, [])]                       # (남은 합, 지금까지 고른 항)
    while stack:
        node, path = stack.pop()
        if node == 0:
            out.append(path)
            continue
        for x in range(node, 0, -1):        # 뒤에서부터 넣어야 1부터 꺼내진다
            stack.append((node - x, path + [x]))
    return out


def paths_bfs(n):
    out = []
    q = deque([(n, [])])
    while q:
        node, path = q.popleft()
        if node == 0:
            out.append(path)
            continue
        for x in range(1, node + 1):
            q.append((node - x, path + [x]))
    return out


n = 4

print('DFS 순서 :', ' , '.join('+'.join(map(str, p)) for p in paths_dfs(n)))
print('BFS 순서 :', ' , '.join('+'.join(map(str, p)) for p in paths_bfs(n)))

## 문제 2 &nbsp;&nbsp;`비선형 탐색`

강인이가 계단을 오르려고 한다. 계단은 모두 `n` 칸으로 구성되어 있다.
강인이는 한 번에 1칸 또는 2칸을 오를 수 있다.
강인이가 0번째 칸에서 출발하여 `n` 번째 칸으로 올라가는 서로 다른 방법의 수를 구해 보자.

②가 없으면 `c` 가 `n` 을 지나쳐도 멈추지 않아 `RecursionError` 가 난다.
`n` 이 1~5 일 때 답은 1, 2, 3, 5, 8 — 피보나치수열이다.
전역 `n` 은 읽기만 하므로 `global` 이 필요 없다(교과서 코드의 `global ans` 는 대입하기 때문에 필요하다).

**실행 결과**
```
n = 5
방법의 수 : 8
```

In [ ]:
def climb(c):
    if c == n:         # ①
        return 1

    if c > n:          # ②
        return 0

    return climb(c + 1) + climb(c + 2)      # ③


n = 5

print('n =', n)
print('방법의 수 :', climb(0))

### 교과서 코드 (교과서 176쪽 · 지도서 175쪽 정답)

```
입력          실행 결과
5             8
```

In [ ]:
ans = 0
def f(c):
  if c==n:
    global ans
    ans = ans+1
    return
  if c>n:
    return
  f(c+1)
  f(c+2)

n = int(input())
f(0)
print(ans)

### 점화식 그대로 쓰면 &nbsp;&nbsp;<span style="color:#6b4bc4">남은 칸 수로 재귀</span>

매개변수를 **지금 서 있는 칸** `c` 대신 **남은 칸 수** `k` 로 잡으면 `f(n) = f(n-1) + f(n-2)` 가 글자 그대로 코드가 된다.
남은 칸이 1이어도 2칸을 오를 수 있으므로 `k < 0` 조건은 여전히 필요하다.

**실행 결과**
```
n = 5
방법의 수 : 8
```

In [ ]:
def climb(k):              # k = 남은 칸 수
    if k == 0:             # ① 딱 맞게 도착
        return 1

    if k < 0:              # ② 지나쳤다
        return 0

    return climb(k - 1) + climb(k - 2)      # ③ f(n) = f(n-1) + f(n-2)


n = 5

print('n =', n)
print('방법의 수 :', climb(n))

### 재귀를 쓰지 않으면 &nbsp;&nbsp;<span style="color:#6b4bc4">반복문 풀이</span>

n번째 칸에 오는 방법은 **n−1번 칸에서 1칸** 또는 **n−2번 칸에서 2칸** 뿐이다. 곧 `f(n) = f(n-1) + f(n-2)`.

**실행 결과**
```
n = 5
반복문으로 구한 방법의 수 : 8
```

In [ ]:
n = 5

f = [0] * (n + 1)
f[0] = 1
f[1] = 1
for i in range(2, n + 1):
    f[i] = f[i-1] + f[i-2]

print('n =', n)
print('반복문으로 구한 방법의 수 :', f[n])

### 재귀를 쓰지 않으면 &nbsp;&nbsp;<span style="color:#6b4bc4">DFS · BFS 풀이</span>

문제 1과 **같은 틀**이다. 뿌리는 `0`(출발 칸), 자식은 `c + 1`, `c + 2` 이다.
문제 1과 달리 자식이 언제나 2개인 이진 트리이고, 지나쳐 버리는 실패한 잎(`c > n`)이 생긴다.

```
              0
          ┌───┴───┐
          1       2
        ┌─┴─┐   ┌─┴─┐
        2   3   3   4 ✓
      ┌─┴┐ ┌┴─┐ ┌┴─┐
      3  4✓4✓ 5✗ 4✓5✗
     ┌┴┐
     4✓5✗
```

**실행 결과**
```
n = 5
DFS(스택)로 구한 방법의 수 : 8
BFS(큐)로 구한 방법의 수   : 8
```

In [ ]:
from collections import deque

def climb_dfs(n):
    stack = [0]                     # 지금 서 있는 칸 번호
    cnt = 0
    while stack:
        c = stack.pop()             # ① 맨 뒤에서 꺼낸다 (LIFO)
        if c == n:                  # 정확히 도착 — 방법 하나
            cnt = cnt + 1
            continue
        if c > n:                   # ② 지나쳤다 — 이 갈래는 버린다
            continue
        for step in (1, 2):         # 한 번에 1칸 또는 2칸
            stack.append(c + step)
    return cnt


def climb_bfs(n):
    q = deque([0])
    cnt = 0
    while q:
        c = q.popleft()             # ③ 맨 앞에서 꺼낸다 (FIFO)
        if c == n:
            cnt = cnt + 1
            continue
        if c > n:
            continue
        for step in (1, 2):
            q.append(c + step)
    return cnt


n = 5

print('n =', n)
print('DFS(스택)로 구한 방법의 수 :', climb_dfs(n))
print('BFS(큐)로 구한 방법의 수   :', climb_bfs(n))

#### 오른 방법을 같이 들고 다니면 &nbsp;&nbsp;<span style="color:#6b4bc4">최소 횟수까지 나온다</span>

BFS의 레벨은 오른 횟수이므로, BFS가 처음 찾은 방법이 곧 최소 횟수다(`n = 5` 이면 `1+2+2`, 3번).
DFS가 처음 찾는 것은 `1+1+1+1+1`, 가장 긴 방법이다.

**실행 결과**
```
DFS 순서 : 1+1+1+1+1 , 1+1+1+2 , 1+1+2+1 , 1+2+1+1 , 1+2+2 , 2+1+1+1 , 2+1+2 , 2+2+1
BFS 순서 : 1+2+2 , 2+1+2 , 2+2+1 , 1+1+1+2 , 1+1+2+1 , 1+2+1+1 , 2+1+1+1 , 1+1+1+1+1

BFS가 가장 먼저 찾은 방법 : 1+2+2 -> 3 번 만에 도착
```

In [ ]:
from collections import deque

def steps_dfs(n):
    out = []
    stack = [(0, [])]                       # (지금 서 있는 칸, 지금까지 오른 방법)
    while stack:
        c, path = stack.pop()
        if c == n:
            out.append(path)
            continue
        if c > n:                           # 지나친 갈래는 버린다
            continue
        for step in (2, 1):                 # 거꾸로 넣어야 1칸부터 꺼내진다
            stack.append((c + step, path + [step]))
    return out


def steps_bfs(n):
    out = []
    q = deque([(0, [])])
    while q:
        c, path = q.popleft()
        if c == n:
            out.append(path)
            continue
        if c > n:
            continue
        for step in (1, 2):
            q.append((c + step, path + [step]))
    return out


n = 5

print('DFS 순서 :', ' , '.join('+'.join(map(str, p)) for p in steps_dfs(n)))
print('BFS 순서 :', ' , '.join('+'.join(map(str, p)) for p in steps_bfs(n)))

first = steps_bfs(n)[0]                     # BFS가 가장 먼저 찾은 것 = 최소 횟수
print()
print('BFS가 가장 먼저 찾은 방법 :', '+'.join(map(str, first)), '->', len(first), '번 만에 도착')

## 문제 3 &nbsp;&nbsp;`선형 탐색`

100 이하의 자연수 `n` 이 주어질 때, 1부터 `n` 까지 자연수 중 소수의 개수를 출력해 보자.

①이 없으면 `range(2, 1)` 이 빈 범위라 1이 `True` 가 되어 답이 5가 된다.
약수를 하나만 찾으면 바로 `return False` 로 끝낸다.

**실행 결과**
```
n = 10
소수의 개수 : 4
```

In [ ]:
def is_prime(n):
    if n < 2:                  # ①
        return False

    for d in range(2, n):      # ②
        if n % d == 0:
            return False

    return True


n = 10

count = 0
for x in range(1, n + 1):
    if is_prime(x):            # ③
        count = count + 1

print('n =', n)
print('소수의 개수 :', count)

### 교과서 코드 (교과서 181쪽)

답은 맞지만, 약수를 찾아도 `cnt = 1` 만 하고 안쪽 반복을 끝까지 돈다.

```
입력          실행 결과
10            4
```

In [ ]:
ans = 0
n = int(input())

for i in range(2, n+1):
  cnt = 0
  for j in range(2, i):
     if (i%j)==0:
       cnt = 1
  if cnt==0:
     ans = ans+1

print(ans)

#### 교과서 코드 고치기 ① &nbsp;&nbsp;<span style="color:#6b4bc4">`break` 로 바로 멈추기</span>

약수를 찾으면 `cnt = 1` 다음에 **`break`** 한다. 하나만 찾아도 소수가 아님이 확정되기 때문이다.
`break` 는 가장 안쪽 반복문 하나만 빠져나간다. (함수가 아니므로 `return` 은 쓸 수 없다.)

```
입력          실행 결과
10            4
```

In [ ]:
ans = 0
n = int(input())

for i in range(2, n+1):
  cnt = 0
  for j in range(2, i):
     if (i%j)==0:
       cnt = 1
       break          # 약수를 찾았으니 더 나눠 볼 필요 없다
  if cnt==0:
     ans = ans+1

print(ans)

#### 교과서 코드 고치기 ② &nbsp;&nbsp;<span style="color:#6b4bc4">√i 까지만 나눠 보기</span>

약수는 **짝**으로 나오고 작은 쪽은 언제나 √i 이하다. 그래서 `j*j <= i` 인 동안만 본다.
`while` 로 바꾸면 `j = j+1` 을 직접 써야 한다. 빠뜨리면 무한 반복이다.

```
입력          실행 결과
10            4
```

In [ ]:
ans = 0
n = int(input())

for i in range(2, n+1):
  cnt = 0
  j = 2
  while j*j <= i:      # j 가 √i 이하인 동안만
     if (i%j)==0:
       cnt = 1
       break
     j = j+1
  if cnt==0:
     ans = ans+1

print(ans)

## 문제 4 &nbsp;&nbsp;`선형 탐색`

100 이하의 자연수 `n` 과 `k` 가 주어질 때 1부터 `n` 까지 자연수 중 `k` 와 서로소인 수의 개수를 계산해 보자.

두 수가 **서로소**라는 것은 최대공약수가 1이라는 뜻이다. 1은 모든 수와 서로소다.
`a < b` 여도 첫 회전에서 자동으로 뒤바뀌므로 크기 비교가 필요 없다.

**실행 결과**
```
n = 10 , k = 3
서로소인 수의 개수 : 7
```

In [ ]:
def gcd(a, b):
    while b != 0:          # ①
        a, b = b, a % b    # ② 반드시 동시에 바꾼다
    return a


n = 10
k = 3

count = 0
for x in range(1, n + 1):
    if gcd(x, k) == 1:     # ③
        count = count + 1

print('n =', n, ', k =', k)
print('서로소인 수의 개수 :', count)

### 동시 대입 대신 &nbsp;&nbsp;<span style="color:#6b4bc4">임시 변수로 바꾸기</span>

`a, b = b, a % b` 는 오른쪽을 **원래 값으로 다 계산한 뒤** 한꺼번에 넣는다. 동시 대입이 없는 언어에서는 임시 변수 `r` 을 쓴다.
`a = b`, `b = a % b` 두 줄로 쓰면 `gcd(48, 18)` 이 6 대신 18이 되어 틀린다.

**실행 결과**
```
gcd(48, 18) — 동시 대입 : 6 , 임시 변수 : 6
gcd(4, 9)   — 동시 대입 : 1 , 임시 변수 : 1
gcd(12, 30) — 동시 대입 : 6 , 임시 변수 : 6
```

In [ ]:
def gcd_temp(a, b):
    while b != 0:
        r = a % b          # 원래 a, b 로 나머지를 먼저 계산해 둔다
        a = b
        b = r
    return a


print('gcd(48, 18) — 동시 대입 :', gcd(48, 18), ', 임시 변수 :', gcd_temp(48, 18))
print('gcd(4, 9)   — 동시 대입 :', gcd(4, 9), ', 임시 변수 :', gcd_temp(4, 9))
print('gcd(12, 30) — 동시 대입 :', gcd(12, 30), ', 임시 변수 :', gcd_temp(12, 30))

### 교과서 코드 (교과서 182쪽 · 지도서 184쪽 정답)

```
입력          실행 결과
10 3          7
```

In [ ]:
import math
n, k = map(int, input().split())
ans = 0

for i in range(1, n+1):
  
  if math.gcd(i, k)==1:
    ans = ans+1

print(ans)

### 반복문을 쓰지 않으면 &nbsp;&nbsp;<span style="color:#6b4bc4">재귀 풀이</span>

`gcd(a, b) = gcd(b, a % b)` 를 그대로 옮기면 재귀가 된다. `while` 버전은 이 식을 풀어 쓴 것이다.
`(1 if gcd(n, k) == 1 else 0)` 은 조건이 참이면 1, 거짓이면 0이 되는 조건 표현식이다.

**실행 결과**
```
n = 10,  k = 3  -> 7
n = 100, k = 12 -> 33
```

In [ ]:
def gcd(a, b):
    if b == 0:
        return a
    return gcd(b, a % b)


def count_coprime(n, k):
    if n == 0:
        return 0
    return count_coprime(n - 1, k) + (1 if gcd(n, k) == 1 else 0)


print('n = 10,  k = 3  ->', count_coprime(10, 3))
print('n = 100, k = 12 ->', count_coprime(100, 12))

## 문제 5 &nbsp;&nbsp;`비선형 탐색`

주어진 7개의 수 `1 3 6 7 11 15 18` 에서 3개를 선택하여 더할 때
얻을 수 있는 수 중 3의 배수인 수의 개수를 구하는 프로그램을 작성해 보자.

7개에서 3개를 고르는 방법은 <sub>7</sub>C<sub>3</sub> = 35가지다. 교과서와 같이 **재귀**로 해결해 보자.

`f(i + 1, ...)` 의 `+1` 덕분에 같은 수를 다시 고르거나 순서만 다른 조합을 세지 않는다.
답은 조건을 만족하는 **경우의 수** 12다(`1+11+15` 와 `3+6+18` 처럼 합이 같아도 따로 센다).

**실행 결과**
```
3의 배수인 수의 개수 : 12
```

In [ ]:
A = [1, 3, 6, 7, 11, 15, 18]
N = len(A)
count = 0


def f(start, sum, picked):
    global count

    if picked == 3:                  # ① 3개를 다 골랐다
        if sum % 3 == 0:
            count = count + 1
        return

    for i in range(start, N):        # ② 다음은 i 다음 수부터 고른다
        f(i + 1, sum + A[i], picked + 1)

f(0, 0, 0)
print('3의 배수인 수의 개수 :', count)

### 교과서 코드 (교과서 183쪽 · 지도서 184쪽 정답)

교과서는 `A` 앞에 `0` 을 넣어 1번 인덱스부터 쓰고 `range(k+1, 8)` 로 돈다. 구조는 위 코드와 같다.

```
실행 결과
12
```

In [ ]:
A = [0, 1, 3, 6, 7, 11, 15, 18]
ans = 0

def f(k, s, j) :
  if j==3:
    if (s%3)==0:
      global ans
      ans = ans+1
    return

  for i in range(k+1, 8):
    f(i, s+A[i], j+1)
  return

f(0, 0, 0)
print(ans)

### 재귀를 쓰지 않으면 &nbsp;&nbsp;<span style="color:#6b4bc4">삼중 반복문</span>

고르는 개수가 3개로 고정이라 반복문 세 개로도 된다. `i < j < k` 가 재귀의 `i + 1` 역할이다.
다만 고르는 개수가 바뀌면 반복문의 개수도 바뀌어야 하므로, 일반적인 경우에는 재귀가 필요하다.

**실행 결과**
```
삼중 반복문으로 구한 개수 : 12
```

In [ ]:
A = [1, 3, 6, 7, 11, 15, 18]
N = len(A)

count = 0
for i in range(N):
    for j in range(i + 1, N):
        for k in range(j + 1, N):
            if (A[i] + A[j] + A[k]) % 3 == 0:
                count = count + 1

print('삼중 반복문으로 구한 개수 :', count)

### 재귀를 쓰지 않으면 &nbsp;&nbsp;<span style="color:#6b4bc4">DFS · BFS 풀이</span>

문제 1·2와 **같은 틀**이다. 교과서의 `f(k, s, j)` 가 곧 상태이므로 그대로 상자에 넣고 꺼낸다.
상태에 넣은 `start`(다음에 볼 위치)가 재귀의 `i + 1` 과 같은 일을 한다.
답이 모두 같은 깊이(3개를 고른 층)에만 있으므로 DFS와 BFS가 찾는 순서도 같다.

**실행 결과**
```
DFS 로 찾은 개수 : 12
BFS 로 찾은 개수 : 12
찾은 순서까지 같은가 : True

처음 다섯 가지
   1 + 3 + 11 = 15
   1 + 6 + 11 = 18
   1 + 11 + 15 = 27
   1 + 11 + 18 = 30
   3 + 6 + 15 = 24
```

In [ ]:
from collections import deque

A = [1, 3, 6, 7, 11, 15, 18]
N = len(A)


def pick_dfs(k):
    out = []
    stack = [(0, 0, [])]                        # (다음에 볼 위치, 지금까지의 합, 고른 수들)
    while stack:
        start, s, picked = stack.pop()          # ① 맨 뒤에서 꺼낸다 (LIFO)
        if len(picked) == k:                    # k개를 다 골랐다
            if s % 3 == 0:
                out.append(picked)
            continue
        for i in range(N - 1, start - 1, -1):   # 거꾸로 넣어야 앞의 수부터 꺼내진다
            stack.append((i + 1, s + A[i], picked + [A[i]]))
    return out


def pick_bfs(k):
    out = []
    q = deque([(0, 0, [])])
    while q:
        start, s, picked = q.popleft()          # ② 맨 앞에서 꺼낸다 (FIFO)
        if len(picked) == k:
            if s % 3 == 0:
                out.append(picked)
            continue
        for i in range(start, N):
            q.append((i + 1, s + A[i], picked + [A[i]]))
    return out


d = pick_dfs(3)
b = pick_bfs(3)
print('DFS 로 찾은 개수 :', len(d))
print('BFS 로 찾은 개수 :', len(b))
print('찾은 순서까지 같은가 :', d == b)
print()

print('처음 다섯 가지')
for p in d[:5]:
    print('  ', ' + '.join(map(str, p)), '=', sum(p))

### 세는 방법을 바꾸면 &nbsp;&nbsp;<span style="color:#6b4bc4">동적 계획법(DP) 풀이</span>

알아야 하는 것은 합의 정확한 값이 아니라 **3으로 나눈 나머지**뿐이다. 그래서

> **dp[고른 개수][합의 나머지] = 그런 경우의 수**

라는 4 × 3 = **12칸** 표만 채우면 된다. 답은 `dp[3][0]`.
`j` 를 거꾸로 도는 것은 방금 넣은 수를 같은 회차에 또 쓰지 않기 위해서다. 마지막 줄 합계 35는 <sub>7</sub>C<sub>3</sub> 와 같다.

**실행 결과**
```
고른 개수 | 나머지 0 | 나머지 1 | 나머지 2 |  합계
       0 |        1 |        0 |        0 |     1
       1 |        4 |        2 |        1 |     7
       2 |        8 |        8 |        5 |    21
       3 |       12 |       13 |       10 |    35

3의 배수인 수의 개수 : 12
3개를 고르는 모든 방법 : 35 (= 7C3)
```

In [ ]:
A = [1, 3, 6, 7, 11, 15, 18]
K = 3


def dp_count(A, K):
    dp = [[0] * 3 for _ in range(K + 1)]        # dp[고른 개수][합의 나머지]
    dp[0][0] = 1                                # 아무것도 안 골랐고 합은 0 — 1가지
    for a in A:                                 # 수를 하나씩 넣어 본다
        for j in range(K - 1, -1, -1):          # 거꾸로 돌아야 같은 수를 두 번 안 쓴다
            for r in range(3):
                if dp[j][r] > 0:
                    dp[j + 1][(r + a) % 3] += dp[j][r]
    return dp


dp = dp_count(A, K)

print('고른 개수 | 나머지 0 | 나머지 1 | 나머지 2 |  합계')
for j in range(K + 1):
    print('%8d | %8d | %8d | %8d | %5d'
          % (j, dp[j][0], dp[j][1], dp[j][2], sum(dp[j])))

print()
print('3의 배수인 수의 개수 :', dp[K][0])
print('3개를 고르는 모든 방법 :', sum(dp[K]), '(= 7C3)')